# CPDE-7 Targeted Extraction Testing

This notebook tests the **targeted extraction** methods of `CPDE7LLMService` which extract profiling data ONLY from TARGET messages while using CONTEXT messages for understanding references.

## Why Targeted Extraction?
- Prevents AI assumptions/reflections from becoming profile facts
- Only extracts from what the user actually stated
- Handles confirmation strength (strong "Yes, exactly" vs weak "Maybe")

## Dimensions
1. Core Identity
2. Opinions & Views
3. Preferences & Patterns
4. Desires, Wishes, Hopes & Needs
5. Life Narrative
6. Events & Involvement
7. Entities & Relationships

In [1]:
# Load environment variables
from dotenv import load_dotenv
import os

# Load from project root .env file (6 levels up)
load_dotenv("../../../../../../.env", override=True)

# Verify API key is loaded
print(f"OPENAI_API_KEY loaded: {'Yes' if os.getenv('OPENAI_API_KEY') else 'No'}")

OPENAI_API_KEY loaded: Yes


In [2]:
# Import the CPDE7LLMService and targeted extraction helpers
from chatforge.services.profiling_data_extraction import (
    CPDE7LLMService,
    CPF7_DIMENSIONS,
    format_messages_with_markers,
)
import json

print(f"Available dimensions: {CPF7_DIMENSIONS}")

Available dimensions: ['core_identity', 'opinions_views', 'preferences_patterns', 'desires_needs', 'life_narrative', 'events', 'entities_relationships']


In [3]:
# Initialize the service
cpde_service = CPDE7LLMService(
    provider="openai",
    model_name="gpt-4o-mini",
    temperature=0,
)

print(f"Service initialized: {cpde_service.model_info}")

Service initialized: openai/gpt-4o-mini


In [ ]:
# Helper function to display results
def display_results(result, dimension_name: str):
    """Pretty print extraction results."""
    # Get the dimension result (first attribute) - use class to avoid deprecation warning
    dim_result = getattr(result, list(result.__class__.model_fields.keys())[0])
    
    print(f"\n{'='*60}")
    print(f"{dimension_name.upper()}")
    print(f"{'='*60}")
    print(f"Has content: {dim_result.has_content}")
    print(f"Items extracted: {len(dim_result.items)}")
    
    for i, item in enumerate(dim_result.items, 1):
        print(f"\n--- Item {i} ---")
        print(f"  Source: {item.source_message_id}")
        print(f"  Quote: \"{item.source_quote}\"")
        # Print other fields
        for field_name, field_value in item.model_dump().items():
            if field_name not in ['source_message_id', 'source_quote'] and field_value is not None:
                print(f"  {field_name}: {field_value}")

## Test Conversation

A conversation between a user and an assistant. The targeted extraction will:
- Extract ONLY from user messages (TARGET)
- Use assistant messages for context (CONTEXT)
- Ignore AI assumptions unless strongly confirmed by user

In [5]:
# Conversation with user and assistant messages
# Targeted extraction will ONLY extract from user messages

test_conversation = [
    {"id": "msg_001", "role": "assistant", "content": "Hi! Tell me a bit about yourself."},
    {"id": "msg_002", "role": "user", "content": "Hi! I'm a 34-year-old software engineer living in Seattle. I'm also diabetic."},
    
    {"id": "msg_003", "role": "assistant", "content": "What are your thoughts on remote work?"},
    {"id": "msg_004", "role": "user", "content": "I think remote work is the future, but companies need better async communication tools."},
    
    {"id": "msg_005", "role": "assistant", "content": "You seem like someone who values efficiency over relationships."},
    {"id": "msg_006", "role": "user", "content": "Maybe, I guess."},  # Weak confirmation - should NOT extract
    
    {"id": "msg_007", "role": "assistant", "content": "Do you have any work habits or preferences?"},
    {"id": "msg_008", "role": "user", "content": "I always code better at night and I can't work without my noise-canceling headphones."},
    
    {"id": "msg_009", "role": "assistant", "content": "What are your career goals?"},
    {"id": "msg_010", "role": "user", "content": "I really want to transition into AI/ML and I hope to lead a team someday."},
    
    {"id": "msg_011", "role": "assistant", "content": "It sounds like you're afraid of failure."},
    {"id": "msg_012", "role": "user", "content": "I don't know about that."},  # Denial - should NOT extract
    
    {"id": "msg_013", "role": "assistant", "content": "Tell me about your background."},
    {"id": "msg_014", "role": "user", "content": "I lived in Tokyo for 3 years after college and it completely changed my worldview."},
    
    {"id": "msg_015", "role": "assistant", "content": "What's going on in your life right now?"},
    {"id": "msg_016", "role": "user", "content": "I'm currently interviewing at Google and also dealing with a kitchen renovation that's driving me crazy."},
    
    {"id": "msg_017", "role": "assistant", "content": "Who are the important people in your life?"},
    {"id": "msg_018", "role": "user", "content": "My mentor Sarah from my previous job at Microsoft still helps me weekly. My dog Max keeps me company while I work."},
]

# Format with TARGET/CONTEXT markers
test_messages = format_messages_with_markers(test_conversation, target_roles=["user"])

print("Formatted conversation (TARGET = user, CONTEXT = assistant):")
print("="*60)
print(test_messages)

Formatted conversation (TARGET = user, CONTEXT = assistant):
Message ID: msg_001
Role: assistant (CONTEXT)
Content: Hi! Tell me a bit about yourself.

Message ID: msg_002
Role: user (TARGET)
Content: Hi! I'm a 34-year-old software engineer living in Seattle. I'm also diabetic.

Message ID: msg_003
Role: assistant (CONTEXT)
Content: What are your thoughts on remote work?

Message ID: msg_004
Role: user (TARGET)
Content: I think remote work is the future, but companies need better async communication tools.

Message ID: msg_005
Role: assistant (CONTEXT)
Content: You seem like someone who values efficiency over relationships.

Message ID: msg_006
Role: user (TARGET)
Content: Maybe, I guess.

Message ID: msg_007
Role: assistant (CONTEXT)
Content: Do you have any work habits or preferences?

Message ID: msg_008
Role: user (TARGET)
Content: I always code better at night and I can't work without my noise-canceling headphones.

Message ID: msg_009
Role: assistant (CONTEXT)
Content: What are yo

---
## 1. Core Identity Extraction

In [6]:
print("Extracting Core Identity (TARGETED - user messages only)...")
result_identity = await cpde_service.extract_core_identity_targeted(test_messages)
display_results(result_identity, "Core Identity")

Extracting Core Identity (TARGETED - user messages only)...

CORE IDENTITY
Has content: True
Items extracted: 7

--- Item 1 ---
  Source: msg_002
  Quote: "Hi! I'm a 34-year-old software engineer living in Seattle. I'm also diabetic."
  aspect: age
  state_value: 34 years old

--- Item 2 ---
  Source: msg_002
  Quote: "Hi! I'm a 34-year-old software engineer living in Seattle. I'm also diabetic."
  aspect: profession
  state_value: software engineer

--- Item 3 ---
  Source: msg_002
  Quote: "Hi! I'm a 34-year-old software engineer living in Seattle. I'm also diabetic."
  aspect: location
  state_value: Seattle

--- Item 4 ---
  Source: msg_002
  Quote: "Hi! I'm a 34-year-old software engineer living in Seattle. I'm also diabetic."
  aspect: medical_condition
  state_value: diabetic

--- Item 5 ---
  Source: msg_014
  Quote: "I lived in Tokyo for 3 years after college and it completely changed my worldview."
  aspect: location
  state_value: Tokyo
  temporal: for 3 years after college


/var/folders/3w/9l9mlgsx74xgqtr3v7hy_1kr0000gn/T/ipykernel_74699/777843429.py:5: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  dim_result = getattr(result, list(result.model_fields.keys())[0])


---
## 2. Opinions & Views Extraction

In [7]:
print("Extracting Opinions & Views (TARGETED - user messages only)...")
result_opinions = await cpde_service.extract_opinions_views_targeted(test_messages)
display_results(result_opinions, "Opinions & Views")

Extracting Opinions & Views (TARGETED - user messages only)...

OPINIONS & VIEWS
Has content: True
Items extracted: 3

--- Item 1 ---
  Source: msg_004
  Quote: "I think remote work is the future, but companies need better async communication tools."
  about: remote work
  view: is the future
  qualifier: but companies need better async communication tools

--- Item 2 ---
  Source: msg_010
  Quote: "I really want to transition into AI/ML and I hope to lead a team someday."
  about: AI/ML transition
  view: want to transition
  qualifier: and I hope to lead a team someday

--- Item 3 ---
  Source: msg_014
  Quote: "I lived in Tokyo for 3 years after college and it completely changed my worldview."
  about: living in Tokyo
  view: changed my worldview


/var/folders/3w/9l9mlgsx74xgqtr3v7hy_1kr0000gn/T/ipykernel_74699/777843429.py:5: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  dim_result = getattr(result, list(result.model_fields.keys())[0])


---
## 3. Preferences & Patterns Extraction

In [8]:
print("Extracting Preferences & Patterns (TARGETED - user messages only)...")
result_preferences = await cpde_service.extract_preferences_patterns_targeted(test_messages)
display_results(result_preferences, "Preferences & Patterns")

Extracting Preferences & Patterns (TARGETED - user messages only)...

PREFERENCES & PATTERNS
Has content: True
Items extracted: 2

--- Item 1 ---
  Source: msg_008
  Quote: "I always code better at night"
  activity_category: work
  activity: coding
  preference: better at night

--- Item 2 ---
  Source: msg_008
  Quote: "I can't work without my noise-canceling headphones."
  activity_category: work
  activity: working
  preference: requires noise-canceling headphones


/var/folders/3w/9l9mlgsx74xgqtr3v7hy_1kr0000gn/T/ipykernel_74699/777843429.py:5: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  dim_result = getattr(result, list(result.model_fields.keys())[0])


---
## 4. Desires, Wishes, Hopes & Needs Extraction

In [9]:
print("Extracting Desires & Needs (TARGETED - user messages only)...")
result_desires = await cpde_service.extract_desires_needs_targeted(test_messages)
display_results(result_desires, "Desires & Needs")

Extracting Desires & Needs (TARGETED - user messages only)...

DESIRES & NEEDS
Has content: True
Items extracted: 2

--- Item 1 ---
  Source: msg_010
  Quote: "I really want to transition into AI/ML and I hope to lead a team someday."
  type: want
  target: transition into AI/ML
  is_active: yes
  intensity: really

--- Item 2 ---
  Source: msg_010
  Quote: "I really want to transition into AI/ML and I hope to lead a team someday."
  type: hope
  target: lead a team
  is_active: yes
  temporal: someday


/var/folders/3w/9l9mlgsx74xgqtr3v7hy_1kr0000gn/T/ipykernel_74699/777843429.py:5: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  dim_result = getattr(result, list(result.model_fields.keys())[0])


---
## 5. Life Narrative Extraction

In [10]:
print("Extracting Life Narrative (TARGETED - user messages only)...")
result_narrative = await cpde_service.extract_life_narrative_targeted(test_messages)
display_results(result_narrative, "Life Narrative")

Extracting Life Narrative (TARGETED - user messages only)...

LIFE NARRATIVE
Has content: True
Items extracted: 1

--- Item 1 ---
  Source: msg_014
  Quote: "I lived in Tokyo for 3 years after college and it completely changed my worldview."
  what_happened: lived in Tokyo
  period: 3 years after college
  significance: completely changed my worldview


/var/folders/3w/9l9mlgsx74xgqtr3v7hy_1kr0000gn/T/ipykernel_74699/777843429.py:5: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  dim_result = getattr(result, list(result.model_fields.keys())[0])


---
## 6. Events & Involvement Extraction

In [11]:
print("Extracting Events (TARGETED - user messages only)...")
result_events = await cpde_service.extract_events_targeted(test_messages)
display_results(result_events, "Events & Involvement")

Extracting Events (TARGETED - user messages only)...

EVENTS & INVOLVEMENT
Has content: True
Items extracted: 3

--- Item 1 ---
  Source: msg_002
  Quote: "Hi! I'm a 34-year-old software engineer living in Seattle. I'm also diabetic."
  event: diabetic
  involvement: experiencing

--- Item 2 ---
  Source: msg_016
  Quote: "I'm currently interviewing at Google and also dealing with a kitchen renovation that's driving me crazy."
  event: interviewing at Google
  involvement: candidate
  temporal: currently
  entities_involved: ['Google']

--- Item 3 ---
  Source: msg_016
  Quote: "I'm currently interviewing at Google and also dealing with a kitchen renovation that's driving me crazy."
  event: kitchen renovation
  involvement: experiencing
  temporal: currently
  outcome: driving me crazy


/var/folders/3w/9l9mlgsx74xgqtr3v7hy_1kr0000gn/T/ipykernel_74699/777843429.py:5: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  dim_result = getattr(result, list(result.model_fields.keys())[0])


---
## 7. Entities & Relationships Extraction

In [12]:
print("Extracting Entities & Relationships (TARGETED - user messages only)...")
result_entities = await cpde_service.extract_entities_relationships_targeted(test_messages)
display_results(result_entities, "Entities & Relationships")

Extracting Entities & Relationships (TARGETED - user messages only)...

ENTITIES & RELATIONSHIPS
Has content: True
Items extracted: 2

--- Item 1 ---
  Source: msg_018
  Quote: "My mentor Sarah from my previous job at Microsoft still helps me weekly. My dog Max keeps me company while I work."
  name: Sarah
  entity_type: person
  mentioned_properties: [{'key': 'job', 'value': 'mentor'}, {'key': 'company', 'value': 'Microsoft'}]
  relationship_indicators: ['mentor', 'colleague']
  interaction_metadata: {'frequency': 'weekly', 'context': 'professional', 'recency': None}

--- Item 2 ---
  Source: msg_018
  Quote: "My mentor Sarah from my previous job at Microsoft still helps me weekly. My dog Max keeps me company while I work."
  name: Max
  entity_type: pet
  mentioned_properties: []
  relationship_indicators: ['pet']


/var/folders/3w/9l9mlgsx74xgqtr3v7hy_1kr0000gn/T/ipykernel_74699/777843429.py:5: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  dim_result = getattr(result, list(result.model_fields.keys())[0])


---
## Summary: All 7 Dimensions

In [ ]:
# Collect all results
all_results = {
    'core_identity': result_identity,
    'opinions_views': result_opinions,
    'preferences_patterns': result_preferences,
    'desires_needs': result_desires,
    'life_narrative': result_narrative,
    'events': result_events,
    'entities_relationships': result_entities,
}

# Summary view
print("\n" + "="*60)
print("EXTRACTION SUMMARY")
print("="*60)

total_items = 0
for dim_name, result in all_results.items():
    dim_result = getattr(result, list(result.model_fields.keys())[0])
    count = len(dim_result.items)
    total_items += count
    status = "" if dim_result.has_content else "(empty)"
    print(f"  {dim_name:25} : {count:3} items {status}")

print(f"\n  {'TOTAL':25} : {total_items:3} items")

---
## Using extract_targeted() Method

The service provides `extract_targeted()` to extract selected dimensions from TARGET messages only.

In [ ]:
# Extract all dimensions at once using extract_targeted (from user messages only)
print("Extracting ALL dimensions using extract_targeted()...")
full_result = await cpde_service.extract_targeted(test_conversation, target_roles=["user"])

print("\n" + "="*60)
print("EXTRACT_TARGETED SUMMARY")
print("="*60)

total = 0
for dim_name in CPF7_DIMENSIONS:
    dim_result = getattr(full_result, dim_name, None)
    if dim_result:
        count = len(dim_result.items)
        total += count
        print(f"  {dim_name:25} : {count:3} items")
    else:
        print(f"  {dim_name:25} : None")

print(f"\n  {'TOTAL':25} : {total:3} items")

---
## Using extract_targeted() with Parallel Execution

You can also extract selected dimensions in parallel for better performance.

In [ ]:
# Extract selected dimensions in PARALLEL (targeted)
import time

print("Extracting selected dimensions (core_identity, events, entities_relationships) in PARALLEL...")
start = time.time()
partial_result = await cpde_service.extract_targeted(
    test_conversation,
    target_roles=["user"],
    dimensions=["core_identity", "events", "entities_relationships"],
    parallel=True  # Run concurrently!
)
elapsed = time.time() - start

print(f"\nCompleted in {elapsed:.2f}s")
print("\n" + "="*60)
print("PARALLEL TARGETED EXTRACTION SUMMARY")
print("="*60)

for dim_name in CPF7_DIMENSIONS:
    dim_result = getattr(partial_result, dim_name, None)
    if dim_result:
        count = len(dim_result.items)
        print(f"  {dim_name:25} : {count:3} items")
    else:
        print(f"  {dim_name:25} : (not extracted)")

---
## Using extract_all_7_targeted() - Single LLM Call

Extract all 7 dimensions from TARGET messages only in a single LLM call. More efficient but may be less accurate for complex messages.

In [ ]:
# Extract all 7 dimensions in ONE LLM call (targeted)
import time

print("Extracting ALL 7 dimensions in a SINGLE LLM call (TARGETED)...")
start = time.time()
all_7_result = await cpde_service.extract_all_7_targeted(test_conversation, target_roles=["user"])
elapsed = time.time() - start

print(f"\nCompleted in {elapsed:.2f}s")
print("\n" + "="*60)
print("EXTRACT_ALL_7_TARGETED (SINGLE CALL) SUMMARY")
print("="*60)

total = 0
for dim_name in CPF7_DIMENSIONS:
    dim_result = getattr(all_7_result, dim_name, None)
    if dim_result:
        count = len(dim_result.items)
        total += count
        status = "✓" if dim_result.has_content else "○"
        print(f"  {status} {dim_name:25} : {count:3} items")
    else:
        print(f"  ✗ {dim_name:25} : None")

print(f"\n    {'TOTAL':25} : {total:3} items")
print(f"\n(Single call extracts ONLY from TARGET/user messages)")

In [ ]:
# Show detailed output from extract_all_7 for entities dimension
print("Entities extracted in single-call mode:")
print("="*60)

if all_7_result.entities_relationships.has_content:
    for i, item in enumerate(all_7_result.entities_relationships.items, 1):
        print(f"\n--- Entity {i} ---")
        print(f"  Source: {item.source_message_id}")
        print(f"  Name: {item.name}")
        print(f"  Type: {item.entity_type}")
        print(f"  Properties: {item.mentioned_properties}")
        print(f"  Relationships: {item.relationship_indicators}")
else:
    print("No entities extracted")

---
## Using extract_dimension_targeted() Method

Extract a single dimension by name from TARGET messages only.

In [ ]:
# Extract by dimension name (targeted)
print("Extracting 'events' dimension by name (TARGETED)...")
events_result = await cpde_service.extract_dimension_targeted(test_conversation, "events", target_roles=["user"])
display_results(events_result, "Events (via extract_dimension_targeted)")

---
## Edge Case Testing

Test messages that should NOT produce extractions for certain dimensions.

In [ ]:
# Edge case messages
edge_case_messages = """
Message ID: edge_001
Content: This coffee is cold and the meeting was pointless.

Message ID: edge_002
Content: I had lunch and then went to the store.

Message ID: edge_003
Content: Apple announced a new iPhone yesterday.
"""

print("Edge case messages (should extract minimal data):")
print(edge_case_messages)

In [ ]:
# Test edge cases using extract_all (parallel for speed)
print("Testing edge cases on all 7 dimensions (parallel)...\n")

edge_result = await cpde_service.extract_all(edge_case_messages, parallel=True)

for dim_name in CPF7_DIMENSIONS:
    dim_result = getattr(edge_result, dim_name, None)
    if dim_result:
        count = len(dim_result.items)
        if count == 0:
            print(f"  {dim_name:25} : 0 items (correct!)")
        else:
            print(f"  {dim_name:25} : {count} items (review needed)")
            for item in dim_result.items:
                quote = item.source_quote[:50] + "..." if len(item.source_quote) > 50 else item.source_quote
                print(f"    - {quote}")

---
## Raw JSON Output

In [ ]:
# Show raw JSON for Core Identity
print("Raw JSON output (Core Identity):")
print(json.dumps(result_identity.model_dump(), indent=2))

In [ ]:
# Show raw JSON for Entities
print("Raw JSON output (Entities & Relationships):")
print(json.dumps(result_entities.model_dump(), indent=2))